In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 3-Class Random Forest Classifier (ESI 2, 3, 4) with Before vs. After Calibration Benchmark (`models/rf_all.ipynb`)

This notebook trains a **3-Class Random Forest Classifier** for **ESI 2, 3, and 4** (excluding ESI 1 and ESI 5 rows) incorporating **29 Predictor Features**, **Validation Threshold Temperature Calibration**, and **Before vs. After Performance Comparison Reporting**:

### System Architecture & Workflow
1. **Filtering & Target Definition**: Excludes ESI 1 and ESI 5 rows (`raw_esi %in% c("2", "3", "4")`), targeting a 3-class classification problem (`"2"`, `"3"`, `"4"`).
2. **Predictor Feature Inventory (29 Features)**:
   - **Baseline Features (3)**: `age`, `gender`, `cc_breathingdifficulty`.
   - **10 Binary Vital Anomaly Flags**: `is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`.
   - **16 Continuous Vital Delta & Range Features**: `hr_mean_to_last`, `sbp_mean_to_last`, `spo2_mean_to_last`, `rr_mean_to_last`, `hr_range`, `rr_range`, `spo2_range`, `sbp_range`, `hr_last_to_min`, `rr_last_to_min`, `spo2_last_to_min`, `sbp_last_to_min`, `hr_last_to_max`, `rr_last_to_max`, `spo2_last_to_max`, `sbp_last_to_max`.
3. **Holdout Test Set Partitioning (15%)**: Reserves a dedicated holdout test set (`test_df`, 15%) at the start, kept completely untouched during model selection and validation.
4. **5-Fold Cross-Validation on Validation Set (85%)**: Runs 5-Fold Stratified Cross-Validation on `train_val_df` to measure out-of-fold validation performance.
5. **Threshold Temperature Optimization on Validation Folds**: Evaluates probability calibration temperatures $\tau \in [0.2, 3.0]$ on out-of-fold validation probabilities to minimize Multi-Class Log-Loss, saving `plots/rf_all_threshold_logloss.png`.
6. **Before vs. After Calibration Benchmark**: Evaluates model performance on the Holdout Test Set **Before Calibration** ($	au = 1.0$) vs. **After Calibration** ($	au = 	au^*$) across Log-Loss, Accuracy, Precision, Recall, F1-Score, and ROC-AUC.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  if (requireNamespace("ranger", quietly = TRUE)) {
    library(ranger)
  } else {
    library(randomForest)
  }
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Exclude ESI 1 and 5, Construct 29 Predictor Features
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col
# FILTERING: Remove ESI 1 and ESI 5 rows completely
raw_esi_all <- as.character(raw_df[[target_col]])
keep_mask   <- raw_esi_all %in% c("2", "3", "4")
raw_df  <- raw_df[keep_mask, ]
raw_esi <- raw_esi_all[keep_mask]
cat(sprintf("ESI 1 & 5 Filtering: Removed %d rows (Remaining 3-Class rows [ESI 2, 3, 4]: %d)\n",
            sum(!keep_mask), nrow(raw_df)))
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
# Construct 29 Features: 3 baseline + 10 binary anomaly flags + 16 vital delta & range features
df_full <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0),
  
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
df_full$target_col <- factor(raw_esi, levels = c("2", "3", "4"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_full), nrow(df_full)))
cat(sprintf("3-Class Dataset Ready: %d total rows x %d cols\n", nrow(df_full), ncol(df_full)))
cat("Predictor Features Included (29 Total Features):\n")
print(setdiff(names(df_full), "target_col"))
cat("\nNatural 3-Class Target Distribution ('2', '3', '4'):\n")
print(table(df_full$target_col))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Holdout Test Partitioning (15%) & Standard 5-Fold Cross-Validation on Validation Set (85%)
# ---------------------------------------------------------
set.seed(config$training$random_state)
test_size <- config$training$test_size # 0.15
# Partition into Train/Val Set (85%) and Holdout Test Set (15%)
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
cat("=== Data Partitioning Summary ===\n")
cat(sprintf("Full 3-Class Dataset : %d rows\n", nrow(df_full)))
cat(sprintf("Train/Val Set (85%%)   : %d rows\n", nrow(train_val_df)))
cat(sprintf("Holdout Test Set (15%%): %d rows\n\n", nrow(test_df)))
# Perform 5-Fold CV ONLY on train_val_df (85%)
k_folds <- 5
folds   <- createFolds(train_val_df$target_col, k = k_folds, list = TRUE, returnTrain = FALSE)
oof_val_prob <- matrix(0, nrow = nrow(train_val_df), ncol = 3)
colnames(oof_val_prob) <- c("2", "3", "4")
binary_cols <- c("gender", "cc_breathingdifficulty",
                 "is_dyspnea_total", "is_dyspnea_moderate", "is_bradypnea", "is_tachypnea",
                 "is_hypotension", "is_hypertension", "is_bradycardia_total", "is_bradycardia_moderate",
                 "is_tachycardia_total", "is_tachycardia_moderate")
cont_cols <- setdiff(names(train_val_df), c(binary_cols, "target_col"))
val_fold_accs <- numeric(k_folds)
cat("============================================================\n")
cat(sprintf("   STARTING %d-FOLD CV ON VALIDATION DATASET (85%% DATA)\n", k_folds))
cat("============================================================\n")
is_ranger <- requireNamespace("ranger", quietly = TRUE)
for (k in 1:k_folds) {
  val_idx    <- folds[[k]]
  train_fold <- train_val_df[-val_idx, ]
  val_fold   <- train_val_df[val_idx, ]
  
  preproc_fold <- preProcess(train_fold[, cont_cols, drop = FALSE], method = c("center", "scale"))
  train_fold   <- predict(preproc_fold, train_fold)
  val_fold     <- predict(preproc_fold, val_fold)
  
  if (is_ranger) {
    rf_fold <- ranger::ranger(
      formula     = target_col ~ . ,
      data        = train_fold,
      num.trees   = 100,
      probability = TRUE,
      seed        = config$training$random_state + k,
      verbose     = FALSE
    )
    probs <- predict(rf_fold, data = val_fold)$predictions
  } else {
    rf_fold <- randomForest::randomForest(
      target_col ~ .,
      data  = train_fold,
      ntree = 100
    )
    probs   <- predict(rf_fold, newdata = val_fold, type = "prob")
  }
  
  oof_val_prob[val_idx, ] <- probs
  
  fold_pred_idx <- apply(probs, 1, which.max)
  fold_pred_fac <- factor(colnames(probs)[fold_pred_idx], levels = c("2", "3", "4"))
  fold_cm       <- confusionMatrix(fold_pred_fac, val_fold$target_col)
  fold_acc      <- as.numeric(fold_cm$overall["Accuracy"])
  val_fold_accs[k] <- fold_acc
  
  cat(sprintf("  Validation Fold %d/%d Accuracy: %.4f (%.2f%%)\n",
              k, k_folds, fold_acc, fold_acc * 100))
}
cat("============================================================\n")
cat(sprintf("   5-FOLD CV VALIDATION COMPLETE. Mean Val Accuracy = %.4f (+/- %.4f)\n",
            mean(val_fold_accs), sd(val_fold_accs)))
cat("============================================================\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Threshold Temperature Calibration Optimization & Log-Loss Plotting
# ---------------------------------------------------------
calc_multiclass_logloss <- function(actual_factor, prob_matrix, eps = 1e-15) {
  prob_clamped <- pmin(pmax(prob_matrix, eps), 1 - eps)
  prob_norm    <- prob_clamped / rowSums(prob_clamped)
  classes      <- levels(actual_factor)
  N            <- length(actual_factor)
  
  logloss_sum  <- 0
  for (i in 1:N) {
    act_cls <- as.character(actual_factor[i])
    cls_idx <- match(act_cls, classes)
    logloss_sum <- logloss_sum - log(prob_norm[i, cls_idx])
  }
  return(logloss_sum / N)
}
scale_probabilities <- function(prob_matrix, temp) {
  eps <- 1e-15
  log_p <- log(pmin(pmax(prob_matrix, eps), 1 - eps))
  scaled_logits <- log_p / temp
  exp_logits <- exp(scaled_logits - apply(scaled_logits, 1, max))
  return(exp_logits / rowSums(exp_logits))
}
# Evaluate temperature grid on OOF validation predictions
temp_grid <- seq(0.2, 3.0, by = 0.05)
logloss_results <- numeric(length(temp_grid))
for (idx in seq_along(temp_grid)) {
  t_val <- temp_grid[idx]
  scaled_oof_probs <- scale_probabilities(oof_val_prob, t_val)
  logloss_results[idx] <- calc_multiclass_logloss(train_val_df$target_col, scaled_oof_probs)
}
opt_idx  <- which.min(logloss_results)
opt_temp <- temp_grid[opt_idx]
min_loss <- logloss_results[opt_idx]
base_loss <- calc_multiclass_logloss(train_val_df$target_col, oof_val_prob)
cat("============================================================\n")
cat("   VALIDATION THRESHOLD TEMPERATURE OPTIMIZATION RESULTS\n")
cat("============================================================\n")
cat(sprintf("  Baseline OOF Log-Loss (Temp = 1.0) : %.5f\n", base_loss))
cat(sprintf("  Optimal Temperature (Tau*)         : %.2f\n", opt_temp))
cat(sprintf("  Minimised OOF Log-Loss             : %.5f (Log-Loss Gain: %.5f)\n", min_loss, base_loss - min_loss))
cat("============================================================\n\n")
# Plot Log-Loss vs Calibration Temperature
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
loss_df <- data.frame(Temperature = temp_grid, LogLoss = logloss_results)
p_loss <- ggplot(loss_df, aes(x = Temperature, y = LogLoss)) +
  geom_line(color = "#2b5c8f", linewidth = 1.2) +
  geom_point(aes(x = opt_temp, y = min_loss), color = "#e07a5f", size = 4) +
  geom_vline(xintercept = opt_temp, linetype = "dashed", color = "#e07a5f", linewidth = 0.8) +
  geom_hline(yintercept = min_loss, linetype = "dotted", color = "gray40", linewidth = 0.6) +
  annotate("text", x = opt_temp + 0.35, y = min_loss + 0.02, 
           label = sprintf("Optimal Tau* = %.2f\nMin Log-Loss = %.4f", opt_temp, min_loss), 
           color = "#e07a5f", fontface = "bold", size = 3.8) +
  theme_minimal() +
  labs(title = "Validation Multi-Class Log-Loss vs. Calibration Temperature (Tau)",
       subtitle = "Optimizing probability threshold calibration on 5-Fold out-of-fold predictions",
       x = "Probability Temperature Parameter (Tau)", y = "Multi-Class Log-Loss") +
  theme(plot.title = element_text(face = "bold", size = 13, hjust = 0.5),
        plot.subtitle = element_text(size = 10, hjust = 0.5))
ggsave(file.path(plots_dir, "rf_all_threshold_logloss.png"), plot = p_loss, width = 9, height = 5, dpi = 300)
cat("Threshold Optimization Log-Loss Plot saved to: plots/rf_all_threshold_logloss.png\n")
p_loss

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Final Single Test Set Benchmark (BEFORE vs. AFTER Optimization Report)
# ---------------------------------------------------------
cat("Training final production Random Forest model on train_val_df (85% data)...\n")
preproc_tv <- preProcess(train_val_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_val_scaled <- predict(preproc_tv, train_val_df)
test_scaled      <- predict(preproc_tv, test_df)
if (is_ranger) {
  final_rf <- ranger::ranger(
    formula      = target_col ~ .,
    data         = train_val_scaled,
    num.trees    = 100,
    probability  = TRUE,
    seed         = config$training$random_state,
    verbose      = FALSE
  )
  raw_test_probs <- predict(final_rf, data = test_scaled)$predictions
} else {
  final_rf       <- randomForest::randomForest(
    target_col ~ .,
    data  = train_val_scaled,
    ntree = 100
  )
  raw_test_probs <- predict(final_rf, newdata = test_scaled, type = "prob")
}
colnames(raw_test_probs) <- c("2", "3", "4")
# BEFORE CALIBRATION (Tau = 1.0)
test_pred_raw_idx <- apply(raw_test_probs, 1, which.max)
test_pred_raw_fac <- factor(colnames(raw_test_probs)[test_pred_raw_idx], levels = c("2", "3", "4"))
# AFTER CALIBRATION (Tau = opt_temp)
calib_test_probs  <- scale_probabilities(raw_test_probs, opt_temp)
test_pred_cal_idx <- apply(calib_test_probs, 1, which.max)
test_pred_cal_fac <- factor(colnames(calib_test_probs)[test_pred_cal_idx], levels = c("2", "3", "4"))
act_test_fac <- factor(test_df$target_col, levels = c("2", "3", "4"))
eval_metrics <- function(prob_mat, pred_fac, act_fac) {
  cm   <- confusionMatrix(pred_fac, act_fac)
  acc  <- as.numeric(cm$overall["Accuracy"])
  prec <- as.numeric(cm$byClass[, "Pos Pred Value"])
  rec  <- as.numeric(cm$byClass[, "Sensitivity"])
  prec[is.na(prec)] <- 0
  rec[is.na(rec)]   <- 0
  f1   <- ifelse((prec + rec) > 0, 2 * (prec * rec) / (prec + rec), 0)
  
  roc_auc <- sapply(1:3, function(i) {
    cls_name <- levels(act_fac)[i]
    act_bin  <- ifelse(act_fac == cls_name, 1, 0)
    r_obj    <- tryCatch(pROC::roc(act_bin, prob_mat[, i]), error = function(e) NULL)
    if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
  })
  
  loss <- calc_multiclass_logloss(act_fac, prob_mat)
  
  return(list(
    acc     = acc,
    prec    = prec,
    rec     = rec,
    f1      = f1,
    roc_auc = roc_auc,
    loss    = loss,
    m_prec  = mean(prec),
    m_rec   = mean(rec),
    m_f1    = mean(f1),
    m_roc   = mean(roc_auc, na.rm = TRUE),
    cm      = cm$table
  ))
}
res_before <- eval_metrics(raw_test_probs,   test_pred_raw_fac, act_test_fac)
res_after  <- eval_metrics(calib_test_probs, test_pred_cal_fac, act_test_fac)
# Before vs After Overall Summary Table
before_after_df <- data.frame(
  Metric               = c("Multi-Class Log-Loss", "Overall Accuracy", "Macro Precision", "Macro Recall", "Macro F1-Score", "Macro ROC-AUC"),
  Before_Calibration   = round(c(res_before$loss, res_before$acc, res_before$m_prec, res_before$m_rec, res_before$m_f1, res_before$m_roc), 4),
  After_Calibration    = round(c(res_after$loss,  res_after$acc,  res_after$m_prec,  res_after$m_rec,  res_after$m_f1,  res_after$m_roc), 4),
  Absolute_Delta       = round(c(res_after$loss - res_before$loss, res_after$acc - res_before$acc, res_after$m_prec - res_before$m_prec, res_after$m_rec - res_before$m_rec, res_after$m_f1 - res_before$m_f1, res_after$m_roc - res_before$m_roc), 4)
)
cat(sprintf("============================================================\n"))
cat(sprintf("   HOLDOUT TEST BENCHMARK: BEFORE vs. AFTER CALIBRATION (Tau* = %.2f)\n", opt_temp))
cat(sprintf("============================================================\n"))
print(before_after_df)
cat(sprintf("============================================================\n\n"))
cat("Confusion Matrix BEFORE Calibration (Tau = 1.0):\n")
print(res_before$cm)
cat("\nConfusion Matrix AFTER Calibration (Tau = ", opt_temp, "):\n")
print(res_after$cm)
cat(sprintf("============================================================\n\n"))
# Write CSV Reports
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
write.csv(before_after_df, file = file.path(reports_dir, "rf_all_before_after_calibration_report.csv"), row.names = FALSE)
cat("Before vs After Calibration CSV Report written to: reports/rf_all_before_after_calibration_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Diagnostic Plots (Before vs After Metric Comparison Bar Chart)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
comp_long <- data.frame(
  Metric = rep(c("LogLoss", "Accuracy", "Precision", "Recall", "F1_Score", "ROC_AUC"), 2),
  State  = c(rep("Before Calibration (Tau=1.0)", 6), rep(paste0("After Calibration (Tau=", opt_temp, ")"), 6)),
  Score  = c(res_before$loss, res_before$acc, res_before$m_prec, res_before$m_rec, res_before$m_f1, res_before$m_roc,
             res_after$loss,  res_after$acc,  res_after$m_prec,  res_after$m_rec,  res_after$m_f1,  res_after$m_roc)
)
comp_long$Metric <- factor(comp_long$Metric, levels = c("LogLoss", "Accuracy", "Precision", "Recall", "F1_Score", "ROC_AUC"))
comp_long$State  <- factor(comp_long$State, levels = c("Before Calibration (Tau=1.0)", paste0("After Calibration (Tau=", opt_temp, ")")))
p_comp <- ggplot(comp_long, aes(x = Metric, y = Score, fill = State)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3.2, fontface = "bold") +
  theme_minimal() +
  scale_fill_manual(values = c("Before Calibration (Tau=1.0)" = "#2b5c8f", "After Calibration (Tau=" = "#e07a5f")) +
  scale_fill_brewer(palette = "Set1") +
  labs(title = "Holdout Test Set Performance: Before vs. After Temperature Calibration",
       subtitle = sprintf("Comparing Baseline (Tau=1.0) vs. Optimal Temperature Calibration (Tau*=%.2f)", opt_temp),
       y = "Metric Score / Loss", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13, hjust = 0.5),
        plot.subtitle = element_text(size = 10, hjust = 0.5),
        legend.position = "top")
ggsave(file.path(plots_dir, "rf_all_before_after_comparison.png"), plot = p_comp, width = 10, height = 5, dpi = 300)
cat("Before vs After Metrics Comparison Bar Chart saved to: plots/rf_all_before_after_comparison.png\n")
p_comp

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save Final Production Random Forest Model Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "rf_all_model.rds")
saveRDS(list(model = final_rf, preproc = preproc_tv, is_ranger = is_ranger, opt_temp = opt_temp), file = model_path)
cat("Final Calibrated Random Forest Model saved to:", model_path, "\n")